# Get and save historical weather data for Vienna Marathon dates

Import relevant libraries

In [8]:
!pip install python-dotenv


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from dotenv import load_dotenv
import os
import requests, time
import pandas as pd
from datetime import datetime, timezone


get key for the weather API from .env

In [2]:
load_dotenv()
weather_key = os.getenv("weather_key")


save coordinates for Vienna so that it can be used in the API call

In [3]:
LAT, LON = 48.2082, 16.3738

save relevant marathon dates (from 2000 onwards, 2020 was canceled)

In [4]:
marathon_dates = {
    2000: "2000-05-21", 2001: "2001-05-20", 2002: "2002-05-26",
    2003: "2003-05-25", 2004: "2004-05-16", 2005: "2005-05-22",
    2006: "2006-05-07", 2007: "2007-04-29", 2008: "2008-04-27",
    2009: "2009-04-19", 2010: "2010-04-18", 2011: "2011-04-17",
    2012: "2012-04-15", 2013: "2013-04-14", 2014: "2014-04-13",
    2015: "2015-04-12", 2016: "2016-04-10", 2017: "2017-04-23",
    2018: "2018-04-22", 2019: "2019-04-07",
    # 2020 cancelled
    2021: "2021-09-12", 2022: "2022-04-24", 2023: "2023-04-23",
    2024: "2024-04-21", 2025: "2025-04-06", 2026: "2026-04-19",
}

fetch historical weather on marathon dates

In [11]:
def get_weather(date_str):
    dt = datetime.strptime(date_str, "%Y-%m-%d").replace(hour=9, tzinfo=timezone.utc)
    ts = int(dt.timestamp())

    r = requests.get(
        "https://api.openweathermap.org/data/3.0/onecall/timemachine",  # hardcoded
        params={
            "lat": LAT, "lon": LON,
            "dt": ts,
            "appid": weather_key,
            "units": "metric"
        }
    )
    if r.status_code != 200:
        print(f"  ✗ Error {r.status_code}: {r.json()}")
        return {}

    d = r.json().get("data", [{}])[0]
    return {
        "temp_c":     d.get("temp"),
        "feels_like": d.get("feels_like"),
        "humidity":   d.get("humidity"),
        "wind_kph":   round(d.get("wind_speed", 0) * 3.6, 1),
        "weather":    d.get("weather", [{}])[0].get("description"),
        "clouds":     d.get("clouds"),
        "rain_mm":    d.get("rain", {}).get("1h", 0),
    }

Fetch all relevant dates

In [12]:
results = {}
for year, date in marathon_dates.items():
    print(f"Fetching {year}...", end=" ")
    results[year] = get_weather(date)
    print("✓")
    time.sleep(0.5)


Fetching 2000... ✓
Fetching 2001... ✓
Fetching 2002... ✓
Fetching 2003... ✓
Fetching 2004... ✓
Fetching 2005... ✓
Fetching 2006... ✓
Fetching 2007... ✓
Fetching 2008... ✓
Fetching 2009... ✓
Fetching 2010... ✓
Fetching 2011... ✓
Fetching 2012... ✓
Fetching 2013... ✓
Fetching 2014... ✓
Fetching 2015... ✓
Fetching 2016... ✓
Fetching 2017... ✓
Fetching 2018... ✓
Fetching 2019... ✓
Fetching 2021... ✓
Fetching 2022... ✓
Fetching 2023... ✓
Fetching 2024... ✓
Fetching 2025... ✓
Fetching 2026... ✓


build dataframe and save results in csv

In [13]:
rows = [{"year": year, "date": date, **results.get(year, {})}
        for year, date in marathon_dates.items()]

df = pd.DataFrame(rows).set_index("year")
print(df)
df.to_csv("weather.csv")

            date  temp_c  feels_like  humidity  wind_kph           weather  \
year                                                                         
2000  2000-05-21   14.97       13.70        45       7.6        few clouds   
2001  2001-05-20   16.18       15.13        49       7.2     broken clouds   
2002  2002-05-26   12.43       12.05        89      21.6        light rain   
2003  2003-05-25   24.67       24.76        60      25.2         clear sky   
2004  2004-05-16   13.02       12.00        62      21.6        few clouds   
2005  2005-05-22   21.23       21.11        65      16.8   overcast clouds   
2006  2006-05-07   14.30       13.54        67       7.2        few clouds   
2007  2007-04-29   18.13       17.62        62      18.0  scattered clouds   
2008  2008-04-27   15.00       14.07        58      14.4         clear sky   
2009  2009-04-19   15.69       14.99        64       3.6         clear sky   
2010  2010-04-18   13.29       12.01        51      32.4  scatte